# Part 2: Criteria Reclassification & Rating Standardization

This notebook covers standardizing our 10 spatial criteria into standard 1 (least suitable) to 9 (most suitable) scales:
1. **Load Pre-processed data** from Part 1
2. **Implement Reclassification engine** using AHP threshold rules
3. **Populate and Profile distributions** for each criterion

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

### 1. Run Classification engine
We import `reclassify` from `src.reclassification` and apply it to each of our aligned criteria.

In [ ]:
from src.reclassification import reclassify
from src.config import unclassified_paths, classified_paths

# Note: In an integrated run, `data_stack` is computed dynamically in Part 1.
# Here we apply classification rules to the stack:
classified_stack = {}

for name in unclassified_paths.keys():
    if name in data_stack:
        classified_stack[name] = reclassify(data_stack[name], name)
        print(f"✓ Classified unclassified layer: {name}")

for name in classified_paths.keys():
    if name in data_stack:
        arr = np.nan_to_num(data_stack[name], nan=0).astype(np.uint8)
        arr = np.clip(arr, 0, 9)
        classified_stack[name] = arr
        print(f"✓ Loaded pre-classified layer: {name}")

### 2. Standardized Classes Distribution Profiling
Inspect the pixel distributions to ensure each class is populated correctly.

In [ ]:
class_dist_stats = {}

print("Populated Classes (1-9 Scale) Statistics:")
for name, arr in classified_stack.items():
    total = int(np.sum(arr > 0))
    filled = sum(1 for c in range(1, 10) if np.any(arr == c))
    dist = {}
    print(f"\n{name} ({filled}/9 classes populated)")
    print(f"{'Class':<6} {'Pixels':>12} {'Percentage':>12}")
    print("-" * 35)
    for c in range(1, 10):
        n = int(np.sum(arr == c))
        pct = 100 * n / total if total > 0 else 0
        dist[c] = {'pixels': n, 'pct': round(pct, 2)}
        print(f"{c:<6} {n:>12,} {pct:>11.2f}%")
    class_dist_stats[name] = {
        'total_valid': total,
        'classes_populated': filled,
        'distribution': dist
    }